# Introducción a la Ciencia de Datos: Tarea 2

Este notebook contiene el código de base para realizar la Tarea 2 del curso. Es la continuación de la Tarea 1, por lo que se utilizarán los mismos datos y se puede reutilizar cualquier parte del código de dicha tarea.

Puede copiar este notebook en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y las librerías Pandas y scikit-learn. Para esta tarea se recomienda consultar la sección [Extracting features from text files](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) de la documentación oficial de scikit-learn.

Recuerde que **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook.

In [ ]:
import re
from time import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_validate
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

## Descarga del dataset
Se utilizan los mismos datos que en la Tarea 1. Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train", cache_dir="../data")
df = ds.to_pandas()

# Parte 1: Dataset y representación numérica de texto

## 1. Preparación del dataset
Se utilizará un conjunto de datos reducido de los **tres medios de prensa con mayor cantidad de artículos**.
Se espera que utilice su propia versión de la función `clean_text()` de la Tarea 1.

Particione los datos para generar un conjunto de test del 30% del total, utilizando muestreo estratificado.

**Sugerencia**: utilice el parámetro `stratify` de la función `train_test_split` de scikit-learn y fije también el valor de `random_state` para obtener resultados reproducibles.

In [ ]:
# ── Limpieza previa del dataset ───────────────────────────────────────────────
# Problemas detectados en Tarea 1 que se corrigen antes de filtrar por medio:
#   1) Valores placeholder en 'title': "Not found", "N/A", etc. (strings no nulos)
#   2) Filas con 'article' nulo o placeholder (no hay texto útil)
#   3) Filas con 'publication' nula (no hay etiqueta target)
#   4) Filas duplicadas (mismo title + article)

PLACEHOLDER_RE = re.compile(
    r'^\s*(?:not\s+found|\[not\s+found\]|none|n/?a|nan|-+|unknown|untitled)\s*$',
    re.IGNORECASE,
)

def is_placeholder(val):
    """True si val es NaN o un string que no aporta información real."""
    if pd.isna(val):
        return True
    return bool(PLACEHOLDER_RE.match(str(val).strip()))

n_original = len(df)

mask_pub_null     = df['publication'].apply(is_placeholder)
mask_article_null = df['article'].apply(is_placeholder)
mask_title_ph     = df['title'].apply(is_placeholder)

print(f"Filas originales                   : {n_original:,}")
print(f"  publication nula/placeholder     : {mask_pub_null.sum():,}")
print(f"  article nulo/placeholder         : {mask_article_null.sum():,}")
print(f"  title placeholder (solo reporte) : {mask_title_ph.sum():,}")

# Eliminar filas sin etiqueta target o sin cuerpo de artículo
df_clean = df[~mask_pub_null & ~mask_article_null].copy()

# Eliminar duplicados exactos (mismo title + article)
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['title', 'article'])
n_dup = n_before - len(df_clean)

print(f"\nDuplicados eliminados              : {n_dup:,}")
print(f"Filas finales                      : {len(df_clean):,}")
print(f"Total filas eliminadas             : {n_original - len(df_clean):,} "
      f"({(n_original - len(df_clean)) / n_original * 100:.1f}%)")

In [ ]:
def clean_text(df, column_name):
    """
    Normalización de texto portada de la Tarea 1:
    - Elimina primera línea (cabecera hasta el primer \\n)
    - Minúsculas
    - Elimina URLs (http/https)
    - Elimina puntuación y caracteres no alfanuméricos
    - Elimina números aislados
    - Normaliza espacios
    """
    result = df[column_name].astype(str)

    result = result.str.replace(r"^[^\n]*\n", "", regex=True)
    result = result.str.lower()
    result = result.str.replace(r'https?://\S+', ' ', regex=True)
    result = result.str.replace(r'[^\w\s]', ' ', regex=True)
    result = result.str.replace(r'\b\d+\b', ' ', regex=True)
    result = result.str.replace(r'[\n\r\t]', ' ', regex=True)
    result = result.str.replace(r'\s+', ' ', regex=True)
    result = result.str.strip()

    return result

In [ ]:
pub_counts = df_clean['publication'].value_counts()
top_3_publications = pub_counts.head(3).index.tolist()

print("Top 3 medios de prensa:")
for i, pub in enumerate(top_3_publications, 1):
    print(f"  {i}. {pub}: {pub_counts[pub]:,} artículos")

df_top_3 = df_clean[df_clean['publication'].isin(top_3_publications)].copy()
print(f"\nTotal artículos (top 3): {len(df_top_3):,}")

df_top_3["CleanText"] = clean_text(df_top_3, "article")

mask_empty = df_top_3["CleanText"].str.strip().eq("") | df_top_3["CleanText"].isna()
if mask_empty.sum() > 0:
    print(f"Artículos con texto vacío tras clean_text: {mask_empty.sum()} → eliminados")
    df_top_3 = df_top_3[~mask_empty].copy()
print(f"Total artículos finales: {len(df_top_3):,}")

In [ ]:
X = df_top_3["CleanText"]
y = df_top_3["publication"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Train : {len(X_train):,} artículos  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test  : {len(X_test):,} artículos  ({len(X_test)/len(X)*100:.0f}%)")

print(f"\nDistribución por medio:")
print(pd.DataFrame({
    "Train n": y_train.value_counts().reindex(top_3_publications),
    "Train %": (y_train.value_counts(normalize=True) * 100).round(1).reindex(top_3_publications),
    "Test n":  y_test.value_counts().reindex(top_3_publications),
    "Test %":  (y_test.value_counts(normalize=True) * 100).round(1).reindex(top_3_publications),
}).to_string())

## 2. Verificación del balance de clases
Genere una visualización que permita verificar que el balance de artículos de cada medio es similar en train y test.

In [ ]:
palette = sns.color_palette("tab10", n_colors=3)
pub_to_color = dict(zip(top_3_publications, palette))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (split_name, y_split) in zip(axes, [("Train (70%)", y_train), ("Test (30%)", y_test)]):
    counts = y_split.value_counts().reindex(top_3_publications)
    pcts   = counts / len(y_split) * 100
    bars   = ax.bar(pcts.index, pcts.values, color=palette)
    ax.set_title(f"Balance de clases — {split_name}")
    ax.set_ylabel("Proporción (%)")
    ax.tick_params(axis='x', rotation=15)
    ax.set_ylim(0, pcts.max() * 1.18)
    for bar, v in zip(bars, pcts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.4,
                f"{v:.1f}%", ha='center', va='bottom', fontsize=10)

plt.suptitle("Distribución de artículos por medio en Train vs. Test", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Representación Bag of Words
Transforme el texto del conjunto de entrenamiento a una representación numérica (features) de conteo de palabras (*bag of words*).
Explique brevemente cómo funciona esta técnica y muestre un ejemplo.
En particular, explique el tamaño de la matriz resultante y la razón por la que es una matriz *sparse*.

**Sugerencia**: puede ser útil imaginar qué sucedería con la memoria RAM requerida si no estuviéramos trabajando con un conjunto de datos reducido.

In [ ]:
count_vec = CountVectorizer()
X_train_bow = count_vec.fit_transform(X_train)

vocab_bow = count_vec.get_feature_names_out()

print(f"Dimensiones de la matriz BoW : {X_train_bow.shape}")
print(f"  Filas    : {X_train_bow.shape[0]:,}  (artículos en train)")
print(f"  Columnas : {X_train_bow.shape[1]:,}  (palabras en el vocabulario)")
print(f"\nTipo     : {type(X_train_bow)}")
print(f"Densidad : {X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]) * 100:.3f}%  "
      f"({X_train_bow.nnz:,} valores no-cero de {X_train_bow.shape[0] * X_train_bow.shape[1]:,} posibles)")

dense_mb  = X_train_bow.shape[0] * X_train_bow.shape[1] * 8 / 1e6
sparse_mb = (X_train_bow.data.nbytes + X_train_bow.indices.nbytes + X_train_bow.indptr.nbytes) / 1e6
print(f"\nMemoria estimada si fuera densa  : {dense_mb:,.0f} MB")
print(f"Memoria real (sparse)            : {sparse_mb:.1f} MB")

doc0 = X_train_bow[0].toarray()[0]
top_idx = doc0.argsort()[::-1][:10]
print(f"\nEjemplo — top 10 palabras por conteo en el artículo 0 de train:")
for idx in top_idx:
    if doc0[idx] > 0:
        print(f"  '{vocab_bow[idx]}': {int(doc0[idx])}")

## 4. Representación TF-IDF
Explique brevemente qué es un **n-grama**.
Obtenga la representación numérica *Term Frequency - Inverse Document Frequency* (TF-IDF).
Explique brevemente en qué consiste esta transformación adicional.

In [ ]:
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

vocab_tfidf = tfidf_vec.get_feature_names_out()

print(f"Dimensiones TF-IDF (train) : {X_train_tfidf.shape}")
print(f"  Filas    : {X_train_tfidf.shape[0]:,}")
print(f"  Columnas : {X_train_tfidf.shape[1]:,}  (mismo vocabulario que BoW)")
print(f"\nDimensiones TF-IDF (test)  : {X_test_tfidf.shape}")
print("  (transform aplica el vocabulario del train; no se agregan columnas nuevas)")

doc0_tfidf = X_train_tfidf[0].toarray()[0]
top_idx = doc0_tfidf.argsort()[::-1][:10]
print(f"\nEjemplo — top 10 términos por peso TF-IDF en el artículo 0 de train:")
for idx in top_idx:
    if doc0_tfidf[idx] > 0:
        print(f"  '{vocab_tfidf[idx]}': {doc0_tfidf[idx]:.4f}")

## 5. Visualización PCA sobre TF-IDF
Muestre en un mapa el conjunto de entrenamiento, utilizando las dos primeras componentes PCA sobre los vectores de TF-IDF.
Analice los resultados y compare qué sucede si utiliza:
- a) el filtrado de `stop_words` para idioma inglés;
- b) el parámetro `use_idf=True`;
- c) `ngram_range=(1,2)`.

Opcionalmente, también puede analizar qué sucede si no elimina los signos de puntuación.

¿Se pueden separar los medios de prensa utilizando sólo 2 componentes principales?
Haga una visualización que permita entender cómo varía la varianza explicada a medida que se agregan componentes (por ejemplo, hasta 10 componentes).

Discuta además si la separación observada puede deberse a diferencias de estilo editorial, a diferencias temáticas o a pistas explícitas del medio que no hayan sido removidas en la limpieza.

In [ ]:
# TODO: Aplique PCA con 2 componentes sobre X_train_tfidf y grafique los resultados,
# coloreando los puntos según el medio de prensa.


In [ ]:
# TODO: Compare los resultados de PCA con distintas configuraciones del TfidfVectorizer:


In [ ]:
# TODO: Genere una visualización que muestre cómo varía la varianza explicada
# a medida que se agregan componentes PCA (por ejemplo, hasta 10 componentes).



# Parte 2: Entrenamiento y Evaluación de Modelos

## 1. Multinomial Naive Bayes
Entrene el modelo *Multinomial Naive Bayes* para clasificar los artículos según a qué medio de prensa pertenece el texto.
Utilice dicho modelo para clasificar los artículos del conjunto de test, y reporte el valor de *accuracy* y la **matriz de confusión**.
Reporte además el valor de *precision* y *recall* para cada medio.
Explique cómo se relacionan estos valores con la matriz anterior.

¿Qué problemas puede tener el hecho de mirar solamente el valor de *accuracy*?
Considere qué sucedería con esta métrica si el desbalance de datos fuera aún mayor entre medios.

**Sugerencia**: utilice el método `from_predictions` de `ConfusionMatrixDisplay` para realizar la matriz.

In [ ]:
# TODO: Entrene Multinomial Naive Bayes sobre X_train_tfidf


In [ ]:
# TODO: Evalúe el modelo sobre el conjunto de test


## 2. Validación cruzada y búsqueda de hiperparámetros
Explique cómo funciona la técnica de **validación cruzada** (*cross-validation*).
Implemente una búsqueda de hiperparámetros usando `GridSearchCV`.
Genere una visualización que permita comparar las métricas (por ejemplo, *accuracy*) de los distintos modelos entrenados, viendo el valor promedio y la variabilidad de las mismas en todos los *splits* (por ejemplo, en un gráfico de violín).

In [ ]:
# TODO: Defina la grilla de hiperparámetros y ejecute GridSearchCV


In [ ]:
# TODO: Genere una visualización (por ejemplo, gráfico de violín) que compare
# la accuracy de los distintos modelos entrenados en GridSearchCV,
# mostrando la variabilidad en los distintos splits.


## 3. Entrenamiento final con el mejor modelo
Elija el mejor modelo (mejores parámetros) y vuelva a entrenar sobre todo el conjunto de entrenamiento disponible (sin quitar datos para validación).
Reporte el valor final de las métricas y la matriz de confusión.
Discuta las limitaciones de utilizar un modelo basado en *bag of words* o TF-IDF para el análisis de texto.

In [ ]:
# TODO: Entrene el mejor modelo sobre todo el conjunto de entrenamiento


In [ ]:
# TODO: Evalúe el mejor modelo sobre el conjunto de test y reporte métricas finales


## 4. Modelo alternativo
Evalúe al menos un modelo más (dentro de scikit-learn) aparte de *Multinomial Naive Bayes* para clasificar el texto utilizando las mismas *features* de texto.
Explique brevemente cómo funciona y compare los resultados con el anterior.

In [ ]:
# TODO: Entrene al menos un modelo alternativo de scikit-learn


## 5. Cambio de medio de prensa
Evalúe el problema cambiando al menos un medio de prensa.
En particular, observe el (des)balance de datos y los problemas que pueda generar, así como cualquier indicio que pueda ver en el mapeo previo con PCA.
Puede ser útil comentar acerca de técnicas como sobre-muestreo y submuestreo; no es necesario implementarlas.

In [ ]:
# TODO: Seleccione una combinación diferente de tres medios de prensa (cambiando al menos uno)
# y repita el proceso de entrenamiento y evaluación.

# Analice el balance de clases en este nuevo conjunto de datos
# ...

## 6. Técnica alternativa de extracción de features
Busque información sobre al menos una técnica alternativa de extraer *features* de texto.
Explique brevemente cómo funciona y qué tipo de diferencias esperaría en los resultados.
No se espera que implemente nada en esta parte.

*TODO: Escriba su análisis en el informe.*

## 7. Opcional: Clasificación a nivel de título
Repita la clasificación con los tres medios de prensa originales, pero esta vez clasificando a nivel de **título** en lugar de artículo completo.

In [ ]:
# Opcional: Repita el pipeline de clasificación utilizando el título del artículo
# en lugar del cuerpo del texto.
